# M3 — Type-Specific Semantic State-Transition Boundary Solver

Bounded A0/A1/A2 experiment only. A0 reuses real M1; A1 uses fixed BEFORE→AFTER contrast; A2 adds only the fixed 0.10 adjacent-CLIP motion tie-breaker.

## INPUT CẦN GẮN TRÊN KAGGLE

1. Raw AIC dataset: `/kaggle/input/datasets/nadkli/dataset-aic`
2. MB1 v0.2.2 AI-QC: `/kaggle/input/datasets/irthn1311/triage-eg-mb1-v022-ai-qc-v01`; ZIP nguyên bản hoặc các member Kaggle đã giải nén đều được hỗ trợ và được kiểm tra SHA-256.
3. Notebook 20 candidates: `/kaggle/input/datasets/irthn1311/triage-eg-mb1-v022-candidatess`; hỗ trợ `triage_eg_mb1_v022_candidatess.zip` hoặc thư mục Kaggle đã giải nén chứa `mb1_v022_candidate_manifest.jsonl`.
4. Stage 1B verification: `/kaggle/input/datasets/irthn1311/triage-eg-stage1b-encoder-compatibility-reports`
5. Offline OpenAI CLIP: `/kaggle/input/datasets/irthn1311/aic2026-openai-clip-vit-b32` hoặc alias Kaggle `/kaggle/input/aic2026-openai-clip-vit-b32`; cần source OpenAI CLIP và checkpoint `ViT-B-32.pt`.
6. Optional only: original complete frozen-seed metadata via `AIC_M3_FROZEN_SEED_METADATA`. Without it, the 13 frozen seeds remain unavailable and M3 runs 4 primary + 1 conditional case.

Internet is required only to clone the repository. No model/network download occurs. CLIP uses CUDA when available; OpenCV CPU remains the frozen default raw decoder.

Output ZIP: `/kaggle/working/triage_eg_m3_v01_bundle.zip`


In [ ]:
import inspect
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get("AIC_REPO_URL", "https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git")
REPO_REF = os.environ.get("AIC_REPO_REF", "TRIAGEEG")
REPO_DIR = Path(os.environ.get("AIC_REPO_DIR", "/kaggle/working/AIC2026_TeamPTK_SGU"))
if not (REPO_DIR / ".git").is_dir():
    if REPO_DIR.exists():
        raise RuntimeError(f"Incomplete repository directory: {REPO_DIR}")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
        check=True,
        env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
    )
COMMIT = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()
BRANCH = subprocess.run(
    ["git", "branch", "--show-current"], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()
sys.path.insert(0, str(REPO_DIR / "src"))
print({"resolved_repo": str(REPO_DIR), "HEAD": COMMIT, "branch": BRANCH})

In [ ]:
DATA_INPUT = Path(os.environ.get("AIC_DATA_ROOT", "/kaggle/input/datasets/nadkli/dataset-aic"))
AI_QC_INPUT = Path(
    os.environ.get(
        "AIC_M3_AI_QC_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-mb1-v022-ai-qc-v01"
    )
)
CANDIDATE_INPUT = Path(
    os.environ.get(
        "AIC_MB1_V022_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-mb1-v022-candidatess"
    )
)
STAGE1B_INPUT = Path(
    os.environ.get(
        "AIC_STAGE1B_ROOT",
        "/kaggle/input/datasets/irthn1311/triage-eg-stage1b-encoder-compatibility-reports",
    )
)
CLIP_INPUT = Path(
    os.environ.get(
        "AIC_OPENAI_CLIP_ASSET_ROOT", "/kaggle/input/datasets/irthn1311/aic2026-openai-clip-vit-b32"
    )
)
FROZEN_INPUT = os.environ.get("AIC_M3_FROZEN_SEED_METADATA")
OUTPUT_ROOT = Path("/kaggle/working/triage_eg_m3_v01")
ZIP_PATH = Path("/kaggle/working/triage_eg_m3_v01_bundle.zip")
print(
    {
        "raw": str(DATA_INPUT),
        "ai_qc": str(AI_QC_INPUT),
        "candidates": str(CANDIDATE_INPUT),
        "stage1b": str(STAGE1B_INPUT),
        "clip": str(CLIP_INPUT),
        "optional_frozen_metadata": FROZEN_INPUT,
        "output_zip": str(ZIP_PATH),
    }
)

In [ ]:
SEARCH_ROOT = Path("/kaggle/input")
MAX_DEPTH, MAX_DIRECTORIES = 7, 8000


def bounded_dirs(root):
    queue, visited = [(Path(root), 0)], 0
    while queue:
        current, depth = queue.pop(0)
        if not current.is_dir():
            continue
        visited += 1
        if visited > MAX_DIRECTORIES:
            raise RuntimeError("Input discovery exceeded directory bound")
        yield current
        if depth < MAX_DEPTH:
            queue.extend(
                (child, depth + 1)
                for child in sorted(current.iterdir())
                if child.is_dir() and not child.is_symlink()
            )


def resolve_artifact(hint, archive_filename, extracted_marker):
    hint = Path(hint)
    if hint.is_file() and hint.name == archive_filename:
        return hint.resolve()
    if hint.is_dir() and (hint / archive_filename).is_file():
        return (hint / archive_filename).resolve()
    if hint.is_dir() and (hint / extracted_marker).is_file():
        return hint.resolve()
    roots = [hint] if hint.exists() else [SEARCH_ROOT]
    for root in roots:
        archives = [
            directory / archive_filename
            for directory in bounded_dirs(root)
            if (directory / archive_filename).is_file()
        ]
        archives = sorted(set(path.resolve() for path in archives))
        if len(archives) == 1:
            return archives[0]
        if len(archives) > 1:
            raise RuntimeError(f"Multiple {archive_filename} files found: {archives}")
        extracted = [
            directory
            for directory in bounded_dirs(root)
            if (directory / extracted_marker).is_file()
        ]
        extracted = sorted(set(path.resolve() for path in extracted))
        if len(extracted) == 1:
            return extracted[0]
        if len(extracted) > 1:
            raise RuntimeError(f"Multiple extracted roots for {extracted_marker}: {extracted}")
    raise RuntimeError(
        f"Missing {archive_filename} or extracted marker {extracted_marker} under {hint}"
    )


def resolve_root(hint, marker):
    hint = Path(hint)
    if hint.is_dir() and (hint / marker).is_file():
        return hint.resolve()
    roots = [hint] if hint.exists() else [SEARCH_ROOT]
    matches = []
    for root in roots:
        matches.extend(
            directory for directory in bounded_dirs(root) if (directory / marker).is_file()
        )
        if matches:
            break
    matches = sorted(set(path.resolve() for path in matches))
    if len(matches) != 1:
        raise RuntimeError(f"Expected exactly one root with {marker}; found {matches}")
    return matches[0]


def resolve_dataset(hint):
    hint = Path(hint)
    if (hint / "map-keyframes-aic25-b1/map-keyframes").is_dir() and any(
        hint.glob("Videos_*/video")
    ):
        return hint.resolve()
    candidates = [
        directory
        for directory in bounded_dirs(hint if hint.exists() else SEARCH_ROOT)
        if (directory / "map-keyframes-aic25-b1/map-keyframes").is_dir()
        and any(directory.glob("Videos_*/video"))
    ]
    candidates = sorted(set(path.resolve() for path in candidates))
    if len(candidates) != 1:
        raise RuntimeError(f"Expected exactly one raw dataset root; found {candidates}")
    return candidates[0]


def resolve_clip_asset(hint):
    hint = Path(hint)
    roots = [
        hint,
        Path("/kaggle/input/aic2026-openai-clip-vit-b32"),
        Path("/kaggle/input/datasets/irthn1311/aic2026-openai-clip-vit-b32"),
    ]
    shallow_checkpoints = list(SEARCH_ROOT.glob("*/checkpoint/ViT-B-32.pt"))
    shallow_checkpoints += list(SEARCH_ROOT.glob("datasets/*/*/checkpoint/ViT-B-32.pt"))
    roots.extend(path.parent.parent for path in shallow_checkpoints)
    resolved = []
    for root in sorted(set(path.resolve(strict=False) for path in roots)):
        source_options = [root / "source/openai_clip", root / "openai_clip"]
        checkpoint_options = [root / "checkpoint/ViT-B-32.pt", root / "ViT-B-32.pt"]
        sources = [path for path in source_options if (path / "clip/__init__.py").is_file()]
        checkpoints = [path for path in checkpoint_options if path.is_file()]
        if len(sources) == 1 and len(checkpoints) == 1:
            resolved.append((root, sources[0].resolve(), checkpoints[0].resolve()))
    resolved = sorted(set(resolved))
    if len(resolved) != 1:
        mounted = (
            sorted(path.name for path in SEARCH_ROOT.iterdir()) if SEARCH_ROOT.is_dir() else []
        )
        raise RuntimeError(
            "Offline CLIP asset is missing or ambiguous. Attach Kaggle dataset "
            "aic2026-openai-clip-vit-b32. "
            f"resolved={resolved}; mounted_top_level={mounted}"
        )
    return resolved[0]


DATASET_ROOT = resolve_dataset(DATA_INPUT)
AI_QC_SOURCE = resolve_artifact(
    AI_QC_INPUT,
    "triage_eg_mb1_v022_ai_qc_v01.zip",
    "mb1_v022_ai_qc_new_candidates_v01.jsonl",
)
CANDIDATE_SOURCE = resolve_artifact(
    CANDIDATE_INPUT,
    "triage_eg_mb1_v022_candidatess.zip",
    "mb1_v022_candidate_manifest.jsonl",
)
STAGE1B_ROOT = resolve_root(STAGE1B_INPUT, "encoder/selected_encoder_contract.json")
CLIP_ROOT, CLIP_SOURCE_ROOT, CLIP_CHECKPOINT = resolve_clip_asset(CLIP_INPUT)
os.environ["AIC_OPENAI_CLIP_ASSET_ROOT"] = str(CLIP_ROOT)
os.environ["AIC_OPENAI_CLIP_SOURCE_ROOT"] = str(CLIP_SOURCE_ROOT)
os.environ["AIC_OPENAI_CLIP_CHECKPOINT"] = str(CLIP_CHECKPOINT)
FROZEN_METADATA = Path(FROZEN_INPUT).resolve(strict=True) if FROZEN_INPUT else None
print(
    json.dumps(
        {
            "dataset": str(DATASET_ROOT),
            "ai_qc_source": str(AI_QC_SOURCE),
            "candidate_source": str(CANDIDATE_SOURCE),
            "stage1b": str(STAGE1B_ROOT),
            "clip": str(CLIP_ROOT),
            "clip_source": str(CLIP_SOURCE_ROOT),
            "clip_checkpoint": str(CLIP_CHECKPOINT),
            "frozen_metadata": str(FROZEN_METADATA) if FROZEN_METADATA else None,
        },
        indent=2,
    )
)

In [ ]:
import yaml

policy = yaml.safe_load(
    (REPO_DIR / "configs/retrieval/gpu_g11_frozen_policy.yaml").read_text(encoding="utf-8")
)
print("GPU POLICY (FROZEN)")
print(json.dumps(policy, indent=2))
print("M3 policy: verified CLIP may use CUDA; OpenCV CPU is raw-decoder default; no GPU research.")

In [ ]:
from triage_eg.experiments.moment_m3 import M3InferenceCase, build_case_registry

registry, registry_summary = build_case_registry(
    ai_qc_zip=AI_QC_SOURCE,
    notebook20_candidates_zip=CANDIDATE_SOURCE,
    frozen_seed_metadata=FROZEN_METADATA,
)
assert "accepted_intervals" not in M3InferenceCase.__dataclass_fields__
assert (
    "accepted"
    not in inspect.signature(
        __import__(
            "triage_eg.experiments.moment_m3", fromlist=["solve_state_transition"]
        ).solve_state_transition
    ).parameters
)
print(json.dumps(registry_summary, indent=2))
print("NO_GT_LEAKAGE_INTERFACE=PASS")
if registry_summary["frozen_seed_metadata_unavailable"]:
    print("FROZEN_SEED_METADATA_UNAVAILABLE:", registry_summary["frozen_seed_metadata_unavailable"])

In [ ]:
from triage_eg.experiments.moment_m3 import M3Config, preflight_m3, run_m3

if OUTPUT_ROOT.exists():
    if OUTPUT_ROOT.parent != Path("/kaggle/working"):
        raise RuntimeError(f"Refusing cleanup outside /kaggle/working: {OUTPUT_ROOT}")
    shutil.rmtree(OUTPUT_ROOT)
ZIP_PATH.unlink(missing_ok=True)
CONFIG = M3Config(
    dataset_root=DATASET_ROOT,
    ai_qc_zip=AI_QC_SOURCE,
    notebook20_candidates_zip=CANDIDATE_SOURCE,
    stage1b_root=STAGE1B_ROOT,
    clip_asset_root=CLIP_ROOT,
    output_root=OUTPUT_ROOT,
    frozen_seed_metadata=FROZEN_METADATA,
    seed=2026,
    device=os.environ.get("AIC_CLIP_DEVICE", "auto"),
    batch_size=int(os.environ.get("AIC_CLIP_BATCH_SIZE", "32")),
    build_git_commit=COMMIT,
    branch=BRANCH,
)
PREFLIGHT = preflight_m3(CONFIG)
print(json.dumps(PREFLIGHT, indent=2))
RESULT = run_m3(CONFIG)
print(
    json.dumps(
        {
            "primary": RESULT["metrics_primary"],
            "by_type": RESULT["metrics_by_type"],
            "secondary": RESULT["metrics_secondary"],
            "decision": RESULT["decision"],
        },
        indent=2,
    )
)

In [ ]:
from IPython.display import Image, display

for path in sorted((OUTPUT_ROOT / "review").glob("*.jpg"))[:4]:
    display(Image(filename=str(path)))
montage = OUTPUT_ROOT / "montages/m3_primary_overview.jpg"
if montage.is_file():
    display(Image(filename=str(montage)))

In [ ]:
from zipfile import ZipFile

from triage_eg.experiments.moment_m3 import create_m3_bundle, formal_report_lines

bundle = create_m3_bundle(OUTPUT_ROOT, ZIP_PATH)
with ZipFile(bundle) as archive:
    members = archive.namelist()
assert not any(
    name.lower().endswith((".pt", ".pth", ".bin", ".npy", ".npz", ".mp4")) for name in members
)
print("ZIP members:", len(members), "size_bytes:", bundle.stat().st_size)
for line in formal_report_lines(RESULT, zip_path=bundle):
    print(line)